In [1]:
# NASTAVIT CESTU DLE SVÉHO UVÁŽENÍ ve formátu:
r"Cesta\ke\gdb.gdb"
HOME_GDB = arcpy.env.workspace

# VLOŽIT Z HeightRegulationLineData.gdb všechny tři vrstvy
# Výsledek jsou pak tyto vrstvy:
#     VyskovaRegulaceNaLinii_l
#     VyskovaRegulaceNaLinii_l_Errors

## Rozdělení stavebních čar vrstvou Výšková regulace na linii - rozhraní z CAD

In [2]:
# Převedení linií rozhraní na body, které leží na průniku rozhraní se stavebními čárami
arcpy.analysis.PairwiseIntersect(
    in_features="vyskova_regulace_na_linii_rozhrani;StavebniCara_l",
    out_feature_class=r"memory\rozhrani_body_multipart",
    join_attributes="NO_FID", # nepřenášej původní OBJECTID vrstev, zachovej jen jejich další atributy
    cluster_tolerance=None,
    output_type="POINT" # nechť výsledkem je bodová geometrie
)
# Změna typu geometrie ze single part to multipart - co řádek v atribuové tabulce, to jeden bod
arcpy.management.MultipartToSinglepart(
    in_features="rozhrani_body_multipart",
    out_feature_class=r"memory\rozhrani_body"
)
# samotné řezání stavebních čar rozhraními
arcpy.management.SplitLineAtPoint(
    in_features="StavebniCara_l",
    point_features=r"memory\rozhrani_body",
    out_feature_class=r"memory\st_cary_rozdelene_rozhranim_z_CAD",
    search_radius=0.001 # toto řeší případné drobné posuny vlivem xy resolution, nutno ještě testovat s nastaveným 0,01 XY resolution a 0,01 XY tolerancí
)


<Result 'memory\\st_cary_rozdelene_rozhranim_z_CAD'>

## Přípojení vrstvy dynamických bloků s výškovou regulací na linii rozdělené vrstvou rozhraní

In [3]:
# připojení dynamických bloků výškové regulace k nařezaným stavebním čárám
arcpy.analysis.SpatialJoin(
    target_features="st_cary_rozdelene_rozhranim_z_CAD",
    join_features="VyskovaRegulaceNaLinii_l_CAD",
    out_feature_class=r"memory\kontrolni_SJ_st_cary_rozdelene_rozhranim_z_CAD",
    join_operation="JOIN_ONE_TO_MANY",
    join_type="KEEP_ALL",
    field_mapping='''
    DRUH_SC "DRUH_SC" true true false 10 Text 0 0,First,#,StavebniCara_l,DRUH_SC,0,9;
    DRUH_INFO "DRUH_INFO" true true false 255 Text 0 0,First,#,StavebniCara_l,DRUH_INFO,0,254;
    ID_LOKAL "ID_LOKAL" true false false 2 Short 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,ID_LOKAL,-1,-1;
    VYSKA_VB "VYSKA_VB" true false false 10 Text 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,VYSKA_VB,0,9;
    VYSKA_VB_I "VYSKA_VB_I" true true false 255 Text 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,VYSKA_VB_I,0,254;
    NP_MIN "NP_MIN" true true false 2 Short 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,NP_MIN,-1,-1;
    NP_MAX "NP_MAX" true true false 2 Short 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,NP_MAX,-1,-1;
    NPU_MAX "NPU_MAX" true true false 2 Short 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,NPU_MAX,-1,-1;
    RIMSA_MIN "RIMSA_MIN" true true false 2 Short 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,RIMSA_MIN,-1,-1;
    RIMSA_MAX "RIMSA_MAX" true true false 2 Short 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,RIMSA_MAX,-1,-1;
    VYSKA_MAX "VYSKA_MAX" true true false 2 Short 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,VYSKA_MAX,-1,-1''',
    # field_mapping nutno nastavit dle datového modelu - je třeba přenést jen Typ st. čáry a všechny atributy pro Výšku z výškové regulace na linii (z dyn. bloků)
    match_option="INTERSECT",
    search_radius=None,
    distance_field_name="",
    match_fields=None
)

<Result 'memory\\kontrolni_SJ_st_cary_rozdelene_rozhranim_z_CAD'>

## Dogenerování rozhraní mezi stavebními čárami, kde rozhraní není nakreslené
rozhraní je přidáno k rozhraní z CAD a uloženo do vrstvy rozhrani_st_cary_new_final

In [ ]:

# Výškové atributy dle datového modelu (VYSKOVA_REGULACE_SPOLECNE_ATRIBUTY)
HEIGHT_ATTRS = ["VYSKA_VB", "VYSKA_VB_I", "NP_MIN", "NP_MAX", "NPU_MAX", "RIMSA_MIN", "RIMSA_MAX", "VYSKA_MAX"]

# Filtrace pouze těch atributů, které skutečně existují ve vrstvě po spatial joinu
available_fields = [f.name for f in arcpy.ListFields("kontrolni_SJ_st_cary_rozdelene_rozhranim_z_CAD")]
dissolve_attrs = [a for a in HEIGHT_ATTRS if a in available_fields]

# Statistiky UNIQUE pro všechny výškové atributy (slouží k detekci chyb - více bloků na jednom úseku)
stats_fields = ";".join([f"{a} UNIQUE" for a in dissolve_attrs])

# Spojení stavebních čar do jedné geometrie členěné rozhraním výškové regulace z CAD,
# pokud spolu sousedí části se stejnou výškovou regulací.
# Dissolve podle VŠECH výškových atributů - segmenty s odlišnou regulací se nespojí.
arcpy.management.Dissolve(
    in_features="kontrolni_SJ_st_cary_rozdelene_rozhranim_z_CAD",
    out_feature_class=r"memory\st_cary_dissolve",
    dissolve_field=dissolve_attrs,
    statistics_fields=stats_fields,
    multi_part="SINGLE_PART",
    unsplit_lines="DISSOLVE_LINES",
    concatenation_separator=""
)

# Vybereme ty spojené geometrie, které mají určenou výšku (alespoň jeden výškový atribut není NULL)
where_clause = " OR ".join([f"{a} IS NOT NULL" for a in dissolve_attrs])
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view="st_cary_dissolve",
    selection_type="NEW_SELECTION",
    where_clause=where_clause,
    invert_where_clause=None
)
# těmto vybraným segmentům převeď oba koncové body do bodové vrstvy
arcpy.management.FeatureVerticesToPoints(
    in_features="st_cary_dissolve",
    out_feature_class=r"memory\st_cary_dissolve_s_vyskou_vertices",
    point_location="BOTH_ENDS"
)
# získej jen ty body, které se překrývají alespoň s jedním dalším - tzn. získám jen body, kde se mění typ stavební čáry zároveň s výškou 
arcpy.analysis.Intersect(
    in_features="st_cary_dissolve_s_vyskou_vertices #",
    out_feature_class=r"memory\rozhrani_st_cary",
    join_attributes="ONLY_FID",
    cluster_tolerance=None,
    output_type="INPUT"
)
# poté body, které leží na sobě spoj do jednoho - vzniknou rozhraní výškové regulace, kde se mění i stavební čára
arcpy.management.Dissolve(
    in_features="rozhrani_st_cary",
    out_feature_class=r"memory\rozhrani_st_cary_final",
    dissolve_field=None,
    statistics_fields=None,
    multi_part="SINGLE_PART",
    unsplit_lines="DISSOLVE_LINES",
    concatenation_separator=""
)
# připoj tato rozhraní k těm už explicitně vymezeným v CADu
arcpy.management.Merge(
    inputs="rozhrani_body;rozhrani_st_cary_final",
    output=r"memory\rozhrani_body_all",
    field_mappings=None,
    add_source="NO_SOURCE_INFO",
    field_match_mode="AUTOMATIC"
)


<Result 'memory\\rozhrani_body_all'>

## Finální vznik vrstvy pro výškovou regulaci
již mohu řezat se všemi rozhraními

In [7]:
# finální dissolve původních stavebních čar - vrstva stavebni_cara
arcpy.management.Dissolve(
    in_features="StavebniCara_l",
    out_feature_class=r"memory\st_cary_dissolve_final",
    dissolve_field=None,
    statistics_fields=None,
    multi_part="SINGLE_PART",
    unsplit_lines="DISSOLVE_LINES",
    concatenation_separator=""
)

<Result 'memory\\st_cary_dissolve_final'>

In [8]:
# samotné řezání stavebních čar nyní již všemi rozhraními
# !!! Nunto nastavit toleranci dle přesnosti, na kterou je ukládána souřadnice (zde výchozí 0,0001 m)
arcpy.management.SplitLineAtPoint(
    in_features="st_cary_dissolve_final",
    point_features=r"memory\rozhrani_body_all",
    out_feature_class=r"memory\st_cary_rozdelene_rozhranim_all",
    search_radius=0.001 # toto řeší případné drobné posuny vlivem xy resolution, nutno ještě testovat s nastaveným 0,01 XY resolution a 0,01 XY tolerancí
)

<Result 'memory\\st_cary_rozdelene_rozhranim_all'>

## Dočištění vrstvy tam, kde vzniklo falešné rozhraní v místě počátečního a koncového bodu vrstvy stavební čáry po dissolve


In [9]:
arcpy.management.FeatureVerticesToPoints(
    in_features="st_cary_rozdelene_rozhranim_all",
    out_feature_class=r"memory\st_cary_rozdelene_rozhranim_all_start_end_vertices",
    point_location="BOTH_ENDS"
)
# vybrání těch počátečních bodů, které nekolidují s body rozhraní (ani CADovskými - vrstva vyskova_regulace_na_linii_rohrani - ani vygenerovanými mezi stavebními čárami)
arcpy.management.SelectLayerByLocation(
    in_layer="st_cary_rozdelene_rozhranim_all_start_end_vertices",
    overlap_type="INTERSECT",
    select_features="rozhrani_body_all",
    search_distance=None,
    selection_type="NEW_SELECTION",
    invert_spatial_relationship="INVERT"
)
# vybrání těch segmentů výškové regulace, které kolidují s výše vybranými falešnými rozhraními
arcpy.management.SelectLayerByLocation(
    in_layer="st_cary_rozdelene_rozhranim_all",
    overlap_type="INTERSECT",
    select_features="st_cary_rozdelene_rozhranim_all_start_end_vertices",
    search_distance=None,
    selection_type="NEW_SELECTION",
    invert_spatial_relationship="NOT_INVERT"
)
# tyto vybrané segmenty jsou spojeny a je tak odstraněno falešné rozhraní
arcpy.management.Dissolve(
    in_features="st_cary_rozdelene_rozhranim_all",
    out_feature_class=r"memory\st_cary_rozdelene_rozhranim_all_merge_mimo_rozhrani",
    dissolve_field=None,
    statistics_fields=None,
    multi_part="SINGLE_PART",
    unsplit_lines="DISSOLVE_LINES",
    concatenation_separator=""
)
# jsou vybrány následně segmenty výškové regulace, které leží na výše vybraných a spojených segmentech
arcpy.management.SelectLayerByLocation(
    in_layer="st_cary_rozdelene_rozhranim_all",
    overlap_type="WITHIN",
    select_features="st_cary_rozdelene_rozhranim_all_merge_mimo_rozhrani",
    search_distance=None,
    selection_type="NEW_SELECTION",
    invert_spatial_relationship="NOT_INVERT"
)

<Result 'st_cary_rozdelene_rozhranim_all'>

In [44]:
# tyto segmenty jsou s vrstvy odstraněny a následně celá vrstva výškové regulace je spojena s vrstvou spojených segmentů výškové regulace bez falešných rozhraní
arcpy.management.DeleteRows(
    in_rows="st_cary_rozdelene_rozhranim_all"
)
arcpy.management.Merge(
    inputs="st_cary_rozdelene_rozhranim_all;st_cary_rozdelene_rozhranim_all_merge_mimo_rozhrani",
    output=r"memory\st_cary_rozdelene_rozhranim_final",
    field_mappings=None,
    add_source="NO_SOURCE_INFO",
    field_match_mode="AUTOMATIC"
)

<Result 'memory\\st_cary_rozdelene_rozhranim_final'>

In [45]:
# finální připojení dynamických bloků výškové regulace k nařezaným stavebním čárám
arcpy.analysis.SpatialJoin(
    target_features="st_cary_rozdelene_rozhranim_final",
    join_features="VyskovaRegulaceNaLinii_l_CAD",
    out_feature_class=r"memory\vyskova_regulace_SJ",
    join_operation="JOIN_ONE_TO_MANY",
    join_type="KEEP_ALL",
    field_mapping='''
    DRUH_SC "DRUH_SC" true true false 10 Text 0 0,First,#,StavebniCara_l,DRUH_SC,0,9;
    DRUH_INFO "DRUH_INFO" true true false 255 Text 0 0,First,#,StavebniCara_l,DRUH_INFO,0,254;
    ID_LOKAL "ID_LOKAL" true false false 2 Short 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,ID_LOKAL,-1,-1;
    VYSKA_VB "VYSKA_VB" true false false 10 Text 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,VYSKA_VB,0,9;
    VYSKA_VB_I "VYSKA_VB_I" true true false 255 Text 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,VYSKA_VB_I,0,254;
    NP_MIN "NP_MIN" true true false 2 Short 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,NP_MIN,-1,-1;
    NP_MAX "NP_MAX" true true false 2 Short 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,NP_MAX,-1,-1;
    NPU_MAX "NPU_MAX" true true false 2 Short 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,NPU_MAX,-1,-1;
    RIMSA_MIN "RIMSA_MIN" true true false 2 Short 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,RIMSA_MIN,-1,-1;
    RIMSA_MAX "RIMSA_MAX" true true false 2 Short 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,RIMSA_MAX,-1,-1;
    VYSKA_MAX "VYSKA_MAX" true true false 2 Short 0 0,First,#,VyskovaRegulaceNaLinii_l_CAD,VYSKA_MAX,-1,-1''',
    # opět nunto nastavit dle finálního datového modelu
    match_option="INTERSECT",
    search_radius=None,
    distance_field_name="",
    match_fields=None
)

# nutný dissolve pro identifikaci případných chyb v dynamických blocích - získám segmenty s dvěma a více přiřazenými dyn. bloky
arcpy.management.Dissolve(
    in_features="vyskova_regulace_SJ",
    out_feature_class=fr"{HOME_GDB}\VyskovaRegulaceNaLinii_l",
    dissolve_field="TARGET_FID",
    statistics_fields="ID_LOKAL FIRST;RIMSA_MAX UNIQUE;RIMSA_MAX FIRST",
    multi_part="SINGLE_PART",
    unsplit_lines="DISSOLVE_LINES",
    concatenation_separator=""
)
# další dva kroky jen vybírají chybné segmenty
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view="VyskovaRegulaceNaLinii_l",
    selection_type="NEW_SELECTION",
    where_clause="UNIQUE_RIMSA_MAX > 1",
    invert_where_clause=None
)
arcpy.conversion.ExportFeatures(
    in_features="VyskovaRegulaceNaLinii_l",
    out_features=fr"{HOME_GDB}\VyskovaRegulaceNaLinii_l_Errors",
    where_clause="",
    use_field_alias_as_name="NOT_USE_ALIAS",
    field_mapping=None,
    sort_field=None
)

<Result 'I:\\04_Hall_of_Fame\\13_Pavel_O\\00_Storage\\SharePoint\\20250321_MADASPRU\\Default.gdb\\VyskovaRegulaceNaLinii_l_Errors'>

In [46]:
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view="st_cary_dissolve",
    selection_type="CLEAR_SELECTION"
)

arcpy.management.SelectLayerByAttribute(
    in_layer_or_view="VyskovaRegulaceNaLinii_l",
    selection_type="CLEAR_SELECTION"
)

arcpy.management.SelectLayerByAttribute(
    in_layer_or_view="st_cary_rozdelene_rozhranim_all_start_end_vertices",
    selection_type="CLEAR_SELECTION"
)

arcpy.management.SelectLayerByAttribute(
    in_layer_or_view="st_cary_rozdelene_rozhranim_all",
    selection_type="CLEAR_SELECTION"
)

arcpy.management.Delete(r"memory\rozhrani_body_multipart")
arcpy.management.Delete(r"memory\rozhrani_body")
arcpy.management.Delete(r"memory\st_cary_rozdelene_rozhranim_z_CAD")
arcpy.management.Delete(r"memory\kontrolni_SJ_st_cary_rozdelene_rozhranim_z_CAD")
arcpy.management.Delete(r"memory\st_cary_dissolve")
arcpy.management.Delete(r"memory\st_cary_dissolve_s_vyskou_vertices")
arcpy.management.Delete(r"memory\rozhrani_st_cary")
arcpy.management.Delete(r"memory\rozhrani_st_cary_final")
arcpy.management.Delete(r"memory\rozhrani_body_all")
arcpy.management.Delete(r"memory\st_cary_dissolve_final")
arcpy.management.Delete(r"memory\st_cary_rozdelene_rozhranim_all")
arcpy.management.Delete(r"memory\st_cary_rozdelene_rozhranim_all_start_end_vertices")
arcpy.management.Delete(r"memory\st_cary_rozdelene_rozhranim_all_merge_mimo_rozhrani")
arcpy.management.Delete(r"memory\st_cary_rozdelene_rozhranim_final")
arcpy.management.Delete(r"memory\vyskova_regulace_SJ")



<Result 'true'>